In [ ]:
# 📖 User Guide

Welcome to the **ChestX Multi-Label Chest X-ray Classification** notebook.

This notebook is organized around **two configuration scripts**, allowing you to customize the training pipeline without modifying the main source code.

## Before You Start

* Attach the required Chest X-ray dataset to the notebook.
* Enable **GPU** from **Notebook Settings → Accelerator → GPU** for faster training.
* Run the notebook from top to bottom unless you are resuming from a checkpoint.

---

## Configuration Script 1: Training Configuration

Use this section to configure all training-related parameters.

You can modify settings such as:

* Model architecture
* Image size
* Batch size
* Number of epochs
* Learning rate
* Optimizer
* Learning rate scheduler
* Loss function
* Data augmentation
* Mixed precision (AMP)
* Random seed

These parameters directly affect model performance and training time.

---

## Configuration Script 2: Runtime & Evaluation Configuration

This section controls the notebook's execution behavior.

Configure options such as:

* Dataset paths
* Output directory
* Checkpoint saving
* Resume training
* Validation settings
* Test-Time Augmentation (TTA)
* Inference options
* Prediction thresholds
* Evaluation metrics
* Output file generation

---

## Running the Notebook

1. Configure the parameters in **Configuration Script 1**.
2. Configure runtime options in **Configuration Script 2**.
3. Run all notebook cells in order.
4. Monitor the training and validation metrics after each epoch.
5. The best-performing model will be saved automatically (if enabled).
6. Use the inference section to generate predictions on new images or the test set.

---

## Outputs

After execution, the notebook can generate:

* Trained model checkpoints
* Best model weights
* Training and validation logs
* ROC-AUC and other evaluation metrics
* Prediction files
* Performance visualizations

---

## Tips

* Reduce the **batch size** if you encounter GPU out-of-memory (OOM) errors.
* Larger image sizes generally improve accuracy but require more GPU memory.
* Use mixed precision (AMP) when supported to speed up training.
* For reproducible results, keep the random seed unchanged.
* Resume from a saved checkpoint instead of restarting long training sessions.

Happy experimenting, and feel free to adjust the configurations to find the best-performing model for your hardware and dataset.


In [ ]:
#==============================
# Check Cuda is Available or Not
#==============================


import torch
print(torch.cuda.is_available())

In [ ]:
import os

ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

print(os.path.exists(f"{ROOT}/train_val_list.txt"))
print(os.path.exists(f"{ROOT}/test_list.txt"))

In [ ]:
#================================================
# Dataset.py   //This Dataset Script for Config 1 Training
#                Script.
#================================================




import os
import glob
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image


class ChestXrayDataset(Dataset):

    def __init__(
        self,
        csv_file,
        root_dir,
        file_list,
        transform=None
    ):

        self.transform = transform
        self.root_dir = root_dir

        print("Loading CSV...")
        self.data = pd.read_csv(csv_file)

        print("Loading image list...")
        with open(file_list, "r") as f:
            self.image_names = [
                line.strip()
                for line in f.readlines()
            ]

        print("Finding image files...")

        image_paths = glob.glob(
            os.path.join(root_dir, "**", "*.png"),
            recursive=True
        )

        self.image_dict = {
            os.path.basename(path): path
            for path in image_paths
        }

        print(
            f"Total images found: {len(self.image_dict)}"
        )

        self.labels_list = [
            "Atelectasis",
            "Consolidation",
            "Infiltration",
            "Pneumothorax",
            "Edema",
            "Emphysema",
            "Fibrosis",
            "Effusion",
            "Pneumonia",
            "Pleural_Thickening",
            "Cardiomegaly",
            "Nodule",
            "Mass",
            "Hernia"
        ]

        self.label_to_idx = {
            label: idx
            for idx, label
            in enumerate(self.labels_list)
        }

        print("Building label lookup table...")

        self.label_map = {
            row["Image Index"]:
            row["Finding Labels"]
            for _, row
            in self.data.iterrows()
        }

        print("Encoding labels...")

        self.encoded_labels = {}

        for img_name, labels in self.label_map.items():

            label_vec = torch.zeros(
                len(self.labels_list),
                dtype=torch.float32
            )

            for disease in labels.split("|"):

                if disease in self.label_to_idx:

                    label_vec[
                        self.label_to_idx[disease]
                    ] = 1.0

            self.encoded_labels[
                img_name
            ] = label_vec

        print("Dataset ready!")

    def __len__(self):

        return len(
            self.image_names
        )

    def __getitem__(
        self,
        idx
    ):

        img_name = self.image_names[idx]

        img_path = self.image_dict.get(
            img_name
        )

        if img_path is None:

            raise FileNotFoundError(
                f"Image not found: {img_name}"
            )

        try:

            with Image.open(img_path) as img:

                image = img.convert(
                    "RGB"
                )

        except Exception:

            image = Image.new(
                "RGB",
                (224, 224)
            )

        if self.transform:

            image = self.transform(
                image
            )

        label_vec = self.encoded_labels[
            img_name
        ]

        return image, label_vec


if __name__ == "__main__":

    from torchvision import transforms

    DATASET_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    dataset = ChestXrayDataset(
        csv_file=f"{DATASET_ROOT}/Data_Entry_2017.csv",
        root_dir=DATASET_ROOT,
        file_list=f"{DATASET_ROOT}/train_val_list.txt",
        transform=transform
    )

    print(
        "Dataset size:",
        len(dataset)
    )

    image, label = dataset[0]

    print(
        "Image shape:",
        image.shape
    )

    print(
        "Label vector:",
        label
    )

In [ ]:
# =============================================================
#  Train.py  —  NIH Chest X-Ray14  |  Tuned Run #1
#  Hardware target: 30 GB RAM, 16 GB VRAM (single GPU)
#
#  Tuned config (vs Run #1 which got 0.8252):
#    FOCAL_GAMMA  : 2.0  → 1.0   (gentler focal effect)
#    LR           : 3e-4         (gentler peak LR)
#    Normal BCE Loss
#    TTA
# =============================================================


import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
import seaborn as sns
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True


def main():

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print("Using device:", device)

    print("GPU Count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(torch.cuda.get_device_name(i))

# ==========================
# DATASET PATHS
# ==========================

    ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

    CSV = f"{ROOT}/Data_Entry_2017.csv"

    TRAIN_LIST = f"{ROOT}/train_val_list.txt"
    TEST_LIST = f"{ROOT}/test_list.txt"

# ==========================
# TRANSFORMS
# ==========================

    train_transform = transforms.Compose([
        transforms.Resize((320,320)),
        #transforms.RandomCrop((224,224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(5),
        transforms.ColorJitter(
            brightness=0.1,
            contrast=0.1
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((320,320)),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])
# ==========================
# DATASET
# ==========================

    train_dataset = ChestXrayDataset(
        CSV,
        ROOT,
        TRAIN_LIST,
        train_transform
    )
    
    test_dataset = ChestXrayDataset(
        CSV,
        ROOT,
        TEST_LIST,
        val_transform
    )


# ==========================
# COMPUTE POSITIVE WEIGHTS
# ==========================
    
    print("Computing class weights...")
    
    all_labels = []
    
    for img_name in train_dataset.image_names:
    
        if img_name in train_dataset.encoded_labels:
    
            all_labels.append(
                train_dataset.encoded_labels[img_name].numpy()
            )

    all_labels = np.array(all_labels)
    
    positive_count = all_labels.sum(axis=0)
    
    negative_count = len(all_labels) - positive_count
    
    pos_weight = torch.tensor(
        negative_count / (positive_count + 1e-6),
        dtype=torch.float32
    )

    pos_weight = torch.clamp(
        pos_weight,
        min=1.0,
        max=100.0
    )

    pos_weight = pos_weight.to(device)
    
    print("Class Weights:")
    print(pos_weight)

    print("\nDisease-wise Class Weights\n")
    
    for disease, weight in zip(
        train_dataset.labels_list,
        pos_weight.cpu().numpy()
    ):
        print(
            f"{disease:<20} {weight:.2f}"
        )

# ==========================
# DATALOADER
# ==========================

    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=1
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=1
    )

# ==========================
# MODEL
# ==========================


    #For Resnet34 Model
    model = models.resnet34(
        weights=models.ResNet34_Weights.IMAGENET1K_V1
    )
    model.fc = nn.Linear(
        model.fc.in_features,
        14
    )
    model = model.to(device)


    #For Densenet121 model
    # model = models.densenet121(
    #     weights=models.DenseNet121_Weights.IMAGENET1K_V1
    # )
    # model.classifier = nn.Linear(
    #     model.classifier.in_features,
    #     14
    # )
    # model = model.to(device)


    #For EfficientNetV2-S Model
    # model = models.efficientnet_v2_s(
    #     weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    # )
    # model.classifier[1] = nn.Linear(
    #     model.classifier[1].in_features,
    #     14
    # )
    # model = model.to(device)


    
# ==========================
# LOSS
# ==========================

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=torch.cuda.is_available()
    )

    best_auc = 0
    epochs = 20
    patience = 3
    counter = 0
    
    metrics = []

# ==========================
# TRAINING LOOP
# ==========================

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")

        model.train()

        running_loss = 0

        for images, labels in tqdm(train_loader):

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            with torch.amp.autocast(
                "cuda",
                enabled=torch.cuda.is_available()
            ):

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        print(
            f"Train Loss: {train_loss:.4f}"
        )


# ==========================
# VALIDATION
# ==========================

        model.eval()

        all_labels = []
        all_outputs = []

        with torch.no_grad():

            for images, labels in test_loader:

                images = images.to(device)

                #Original image prediction
                outputs1 = model(images)
                
                # Horizontally flipped image
                flipped_images = torch.flip(
                    images,
                    dims=[3]
                )
                
                outputs2 = model(flipped_images)
                
                # Average predictions
                probs = (
                    torch.sigmoid(outputs1) +
                    torch.sigmoid(outputs2)
                ) / 2


                all_labels.append(
                    labels.numpy()
                )

                all_outputs.append(
                    probs.cpu().numpy()
                )

        all_labels = np.vstack(all_labels)
        all_outputs = np.vstack(all_outputs)

        try:

            val_auc = roc_auc_score(
                all_labels,
                all_outputs,
                average="macro"
            )

            print(
                f"Validation AUC: {val_auc:.4f}"
            )

            metrics.append({
                "epoch": epoch + 1,
                "loss": train_loss,
                "auc": val_auc
            })

            scheduler.step(val_auc)

            if val_auc > best_auc:
            
                best_auc = val_auc
            
                counter = 0
            
                state_dict = (
                    model.module.state_dict()
                    if isinstance(model, nn.DataParallel)
                    else model.state_dict()
                )

                torch.save({
                    "epoch": epoch + 1,
                    "auc": val_auc,
                    "model_state_dict": state_dict
                },
                "/kaggle/working/best_model.pth")
            
                print(
                    f"Best model saved! AUC={val_auc:.4f}"
                )

            else:
            
                counter += 1
            
                print(
                    f"No improvement ({counter}/{patience})"
                )
            
                if counter >= patience:
            
                    print(
                        "Early stopping triggered!"
                    )
            
                    break
        except Exception as e:

            print(
                f"AUC Error: {e}"
            )


    pd.DataFrame(metrics).to_csv(
        "/kaggle/working/training_metrics.csv",
        index=False
    )
    
    print(
        "Metrics CSV saved."
    )
    print(
        f"\nBest AUC: {best_auc:.4f}"
    )


    disease_names = [
        "Atelectasis",
        "Consolidation",
        "Infiltration",
        "Pneumothorax",
        "Edema",
        "Emphysema",
        "Fibrosis",
        "Effusion",
        "Pneumonia",
        "Pleural_Thickening",
        "Cardiomegaly",
        "Nodule",
        "Mass",
        "Hernia"
    ]

    for i, disease in enumerate(disease_names):
    
        try:
    
            disease_auc = roc_auc_score(
                all_labels[:, i],
                all_outputs[:, i]
            )
    
            print(
                f"{disease}: {disease_auc:.4f}"
            )
    
        except Exception as e:
            print(f"{disease}: {e}")


    metrics_df = pd.DataFrame(metrics)

# ==========================
# LOSS CURVE
# ==========================
    
    plt.figure(figsize=(8, 5))
    
    plt.plot(
        metrics_df["epoch"],
        metrics_df["loss"],
        marker="o"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss Curve")
    plt.grid(True)
    
    plt.savefig(
        "/kaggle/working/loss_curve.png",
        bbox_inches="tight"
    )
    
    plt.close()

# ==========================
# AUC CURVE
# ==========================
    
    plt.figure(figsize=(8, 5))
    
    plt.plot(
        metrics_df["epoch"],
        metrics_df["auc"],
        marker="o"
    )

    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.title("Validation AUC Curve")
    plt.grid(True)
    
    plt.savefig(
        "/kaggle/working/auc_curve.png",
        bbox_inches="tight"
    )

    plt.close()
    
    print("Loss curve saved.")
    print("AUC curve saved.")


# ==========================
# ROC CURVE
# ==========================
    
    disease_names = [
        "Atelectasis",
        "Consolidation",
        "Infiltration",
        "Pneumothorax",
        "Edema",
        "Emphysema",
        "Fibrosis",
        "Effusion",
        "Pneumonia",
        "Pleural_Thickening",
        "Cardiomegaly",
        "Nodule",
        "Mass",
        "Hernia"
    ]

    plt.figure(figsize=(10, 8))
    
    for i, disease in enumerate(disease_names):
    
        try:
    
            fpr, tpr, _ = roc_curve(
                all_labels[:, i],
                all_outputs[:, i]
            )
    
            roc_auc = auc(fpr, tpr)
    
            plt.plot(
                fpr,
                tpr,
                label=f"{disease} ({roc_auc:.3f})"
            )
    
        except Exception as e:
            print(f"{disease}: {e}")

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )
    
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves - NIH Chest X-ray")
    plt.legend(fontsize=8)
    plt.grid(True)
    
    plt.savefig(
        "/kaggle/working/roc_curves.png",
        bbox_inches="tight"
    )
    
    plt.close()
    
    print("ROC Curves saved.")


    
    # ==========================
    # CONFUSION MATRICES
    # ==========================
    
    threshold = 0.3
    
    for i, disease in enumerate(disease_names):
    
        try:
    
            y_true = all_labels[:, i]
    
            y_pred = (
                all_outputs[:, i] >= threshold
            ).astype(int)
    
            cm = confusion_matrix(
                y_true,
                y_pred
            )

            plt.figure(figsize=(5,4))
    
            sns.heatmap(
                cm,
                annot=True,
                fmt="d",
                cmap="Blues"
            )
    
            plt.title(
                f"{disease} Confusion Matrix"
            )
    
            plt.ylabel("Actual")
    
            plt.xlabel("Predicted")
    
            plt.savefig(
                f"/kaggle/working/{disease}_cm.png",
                bbox_inches="tight"
            )
    
            plt.close()

        except Exception as e:
    
            print(
                f"{disease}: {e}"
            )
    
    print("Confusion matrices saved.")


    print("\nClassification Reports\n")
    for i, disease in enumerate(disease_names):
    
        y_true = all_labels[:, i]
    
        y_pred = (
            all_outputs[:, i] >= threshold
        ).astype(int)
    
        print(f"\n{disease}")
    
        print(
            classification_report(
                y_true,
                y_pred,
                digits=4,
                zero_division=0
            )
        )


#====================
# F1-Score
#====================

    macro_f1 = f1_score(
        all_labels,
        (all_outputs >= threshold).astype(int),
        average="macro",
        zero_division=0
    )
    
    print(f"\nMacro F1 Score: {macro_f1:.4f}")




#====================
# Preciosion & Recall
#====================
    
    macro_precision = precision_score(
        all_labels,
        (all_outputs >= threshold).astype(int),
        average="macro",
        zero_division=0
    )
    
    macro_recall = recall_score(
        all_labels,
        (all_outputs >= threshold).astype(int),
        average="macro",
        zero_division=0
    )
    
    print(
        f"Macro Precision: {macro_precision:.4f}"
    )
    
    print(
        f"Macro Recall: {macro_recall:.4f}"
    )


if __name__ == "__main__":
    main()

In [ ]:
# =============================================================
#  Train.py  —  NIH Chest X-Ray14  |  Tuned Run #2
#  Hardware target: 30 GB RAM, 16 GB VRAM (single GPU)
#
#  Tuned config (vs Run #1 which got 0.8252):
#    FOCAL_GAMMA  : 2.0  → 1.0   (gentler focal effect)
#    LR_HEAD      : 3e-4 → 1.5e-4 (gentler peak LR)
#    DROPOUT      : 0.3  → 0.2   (less aggressive regularisation)
#
#  Kaggle ready — single file, no separate Dataset.py needed.
#  Run order:
#    Cell 1:  !pip install albumentations -q
#    Cell 2:  paste this entire file, run
# =============================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from PIL import Image
import glob
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import albumentations as A
from albumentations.pytorch import ToTensorV2


# =============================================================
#  CONSTANTS
# =============================================================
DISEASE_LABELS = [
    "Atelectasis", "Consolidation", "Infiltration",
    "Pneumothorax", "Edema", "Emphysema", "Fibrosis",
    "Effusion", "Pneumonia", "Pleural_Thickening",
    "Cardiomegaly", "Nodule", "Mass", "Hernia"
]

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


# =============================================================
#  TRANSFORMS  (Albumentations)
# =============================================================
def build_train_transform(img_size: int = 384):
    return A.Compose([
        A.Resize(img_size, img_size),
        # geometric
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(0.0, 0.05),
            scale=(0.92, 1.08),
            rotate=(-10, 10),
            mode=0,                 # cv2.BORDER_CONSTANT
            p=0.6,
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
        # photometric
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
        A.RandomBrightnessContrast(
            brightness_limit=0.15, contrast_limit=0.15, p=0.5
        ),
        A.GaussNoise(std_range=(0.02, 0.1), p=0.2),
        # normalise + tensor
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def build_val_transform(img_size: int = 384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


# =============================================================
#  DATASET
# =============================================================
class ChestXrayDataset(Dataset):
    """
    NIH Chest X-Ray14 dataset.
    Pre-encodes all labels into a (N, 14) float32 tensor for speed.
    """

    def __init__(
        self,
        csv_file: str,
        root_dir: str,
        file_list: str,
        transform=None,
        img_size: int = 384,
    ):
        self.root_dir  = root_dir
        self.img_size  = img_size
        self.transform = transform

        print("[Dataset] Loading CSV …")
        df = pd.read_csv(csv_file)

        print("[Dataset] Loading file list …")
        with open(file_list) as f:
            self.image_names = [l.strip() for l in f if l.strip()]

        print("[Dataset] Indexing image files …")
        image_paths = glob.glob(
            os.path.join(root_dir, "**", "*.png"), recursive=True
        )
        self.image_dict = {os.path.basename(p): p for p in image_paths}
        print(f"[Dataset] Found {len(self.image_dict):,} PNG files")

        label_map    = dict(zip(df["Image Index"], df["Finding Labels"]))
        label_to_idx = {lbl: i for i, lbl in enumerate(DISEASE_LABELS)}

        print("[Dataset] Encoding labels …")
        n = len(self.image_names)
        label_matrix = np.zeros((n, len(DISEASE_LABELS)), dtype=np.float32)

        for row_idx, img_name in enumerate(self.image_names):
            raw = label_map.get(img_name, "No Finding")
            for disease in raw.split("|"):
                col = label_to_idx.get(disease)
                if col is not None:
                    label_matrix[row_idx, col] = 1.0

        self.labels      = torch.from_numpy(label_matrix)
        self.labels_list = DISEASE_LABELS
        print(f"[Dataset] Ready — {n:,} samples, {len(DISEASE_LABELS)} classes")

    def compute_pos_weight(self, clamp_max: float = 100.0) -> torch.Tensor:
        pos = self.labels.sum(dim=0)
        neg = len(self.labels) - pos
        pw  = neg / (pos + 1e-6)
        return pw.clamp(min=1.0, max=clamp_max)

    def __len__(self) -> int:
        return len(self.image_names)

    def __getitem__(self, idx: int):
        img_name = self.image_names[idx]
        img_path = self.image_dict.get(img_name)

        if img_path is None:
            img_np = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            try:
                with Image.open(img_path) as pil_img:
                    img_np = np.array(pil_img.convert("RGB"), dtype=np.uint8)
            except Exception:
                img_np = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)

        if self.transform is not None:
            img_tensor = self.transform(image=img_np)["image"]
        else:
            pil = Image.fromarray(img_np).resize(
                (self.img_size, self.img_size), Image.BILINEAR
            )
            img_tensor = torch.from_numpy(
                np.array(pil, dtype=np.float32) / 255.0
            ).permute(2, 0, 1)

        return img_tensor, self.labels[idx]


# =============================================================
#  CONFIG  —  Tuned Run #2
# =============================================================
CFG = dict(
    ROOT     = "/kaggle/input/datasets/organizations/nih-chest-xrays/data",
    SAVE_DIR = "/kaggle/working",

    IMG_SIZE   = 384,

    EPOCHS     = 30,
    PATIENCE   = 5,
    BATCH_SIZE = 32,

    LR_HEAD      = 1.5e-4,   # was 3e-4
    LR_BACKBONE  = 3e-5,
    WEIGHT_DECAY = 1e-4,

    FOCAL_GAMMA    = 1.0,    # was 2.0
    LABEL_SMOOTH   = 0.05,
    POS_WEIGHT_CAP = 50.0,

    DROPOUT = 0.2,           # was 0.3

    TTA_PASSES = 4,

    NUM_WORKERS = 4,
    PIN_MEMORY  = True,
    PREFETCH    = 2,

    SEED = 42,
)

SAVE_PATH = os.path.join(CFG["SAVE_DIR"], "DenseNet201.pth")


# =============================================================
#  FOCAL LOSS
# =============================================================
class FocalBCELoss(nn.Module):
    def __init__(self, pos_weight: torch.Tensor, gamma: float = 1.0, label_smooth: float = 0.05):
        super().__init__()
        self.gamma        = gamma
        self.label_smooth = label_smooth
        self.register_buffer("pos_weight", pos_weight)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if self.label_smooth > 0:
            targets = targets * (1 - self.label_smooth) + 0.5 * self.label_smooth

        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none"
        )

        if self.gamma > 0:
            probs = torch.sigmoid(logits).detach()
            p_t   = probs * targets + (1 - probs) * (1 - targets)
            bce   = bce * (1 - p_t) ** self.gamma

        return bce.mean()


# =============================================================
#  MODEL
# =============================================================

#For EffiientNetV2-S Model
def build_model(num_classes: int = 14, dropout: float = 0.2) -> nn.Module:
    model = models.efficientnet_v2_s(
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    model = model.to(memory_format=torch.channels_last)
    return model


#For DenseNet121 Model
# def build_model(num_classes: int = 14, dropout: float = 0.2) -> nn.Module:
#     model = models.densenet121(
#         weights=models.DenseNet121_Weights.IMAGENET1K_V1
#     )
#     in_features = model.classifier.in_features
#     model.classifier = nn.Sequential(
#         nn.Dropout(p=dropout, inplace=True),
#         nn.Linear(in_features, num_classes),
#     )
#     return model


# For DenseNet201 Model
# def build_model(num_classes: int = 14, dropout: float = 0.2) -> nn.Module:
#     model = models.densenet201(
#         weights=models.DenseNet201_Weights.IMAGENET1K_V1
#     )
#     in_features = model.classifier.in_features
#     model.classifier = nn.Sequential(
#         nn.Dropout(p=dropout, inplace=True),
#         nn.Linear(in_features, num_classes),
#     )
#     return model


def build_optimizer(model: nn.Module, cfg: dict) -> torch.optim.Optimizer:
    backbone_params = list(model.features.parameters())
    head_params     = list(model.classifier.parameters())
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg["LR_BACKBONE"], "weight_decay": cfg["WEIGHT_DECAY"]},
        {"params": head_params,     "lr": cfg["LR_HEAD"],     "weight_decay": 1e-5},
    ])


# =============================================================
#  TTA + EVALUATION
# =============================================================
@torch.no_grad()
def tta_predict(model: nn.Module, images: torch.Tensor, n_passes: int = 4) -> torch.Tensor:
    preds = torch.sigmoid(model(images))

    if n_passes >= 2:
        preds = preds + torch.sigmoid(model(torch.flip(images, dims=[3])))

    if n_passes >= 4:
        H    = images.shape[2]
        crop = int(H * 0.90)
        pad  = (H - crop) // 2
        c1 = images[:, :, pad:pad+crop, pad:pad+crop]
        c1 = F.interpolate(c1, size=(H, H), mode="bilinear", align_corners=False)
        preds = preds + torch.sigmoid(model(c1))
        c2 = torch.flip(c1, dims=[3])
        preds = preds + torch.sigmoid(model(c2))

    return preds / n_passes


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, tta_passes: int = 4):
    model.eval()
    all_labels, all_probs = [], []

    for images, labels in tqdm(loader, desc="  eval", leave=False):
        # images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        images = images.to(device, non_blocking=True)
        probs  = tta_predict(model, images, n_passes=tta_passes)
        all_labels.append(labels.numpy())
        all_probs.append(probs.cpu().numpy())

    all_labels = np.vstack(all_labels)
    all_probs  = np.vstack(all_probs)

    try:
        macro_auc = roc_auc_score(all_labels, all_probs, average="macro")
    except Exception:
        macro_auc = 0.0

    return macro_auc, all_labels, all_probs


# =============================================================
#  PLOTTING
# =============================================================
def save_training_plots(metrics_df: pd.DataFrame, save_dir: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(metrics_df["epoch"], metrics_df["train_loss"], marker="o", label="Train Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss"); axes[0].grid(True)

    axes[1].plot(metrics_df["epoch"], metrics_df["val_auc"], marker="o", color="green", label="Val AUC")
    axes[1].axhline(y=0.86, color="red", linestyle="--", label="Target 0.86")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUC")
    axes[1].set_title("Validation AUC"); axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_roc_curves(labels: np.ndarray, probs: np.ndarray, save_dir: str) -> None:
    fig, ax = plt.subplots(figsize=(11, 9))
    for i, disease in enumerate(DISEASE_LABELS):
        try:
            fpr, tpr, _ = roc_curve(labels[:, i], probs[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f"{disease} ({roc_auc:.3f})")
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("ROC Curves — NIH Chest X-ray14 (EfficientNetV2-S)")
    ax.legend(fontsize=8); ax.grid(True)
    plt.savefig(os.path.join(save_dir, "roc_curves.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_per_class_auc(labels: np.ndarray, probs: np.ndarray, save_dir: str) -> None:
    aucs, names = [], []
    for i, disease in enumerate(DISEASE_LABELS):
        try:
            a = roc_auc_score(labels[:, i], probs[:, i])
            aucs.append(a); names.append(disease)
        except Exception:
            pass
    pairs = sorted(zip(aucs, names))
    aucs_s, names_s = zip(*pairs)

    colors = ["#E74C3C" if a < 0.75 else "#F39C12" if a < 0.83 else "#27AE60" for a in aucs_s]
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(names_s, aucs_s, color=colors)
    ax.axvline(x=0.86, color="navy", linestyle="--", label="Target 0.86")
    for bar, val in zip(bars, aucs_s):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=9)
    ax.set_xlim(0.6, 1.0); ax.set_xlabel("AUC")
    ax.set_title("Per-Disease AUC — EfficientNetV2-S")
    ax.legend(); ax.grid(axis="x", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "per_disease_auc.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_confusion_matrices(labels: np.ndarray, probs: np.ndarray, save_dir: str, threshold: float = 0.35) -> None:
    for i, disease in enumerate(DISEASE_LABELS):
        try:
            y_true = labels[:, i]
            y_pred = (probs[:, i] >= threshold).astype(int)
            cm = confusion_matrix(y_true, y_pred)
            fig, ax = plt.subplots(figsize=(4, 3))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
            ax.set_title(f"{disease} (thresh={threshold})")
            ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f"cm_{disease}.png"), dpi=100, bbox_inches="tight")
            plt.close(fig)
        except Exception:
            pass


# =============================================================
#  MAIN
# =============================================================
def main():
    torch.manual_seed(CFG["SEED"])
    np.random.seed(CFG["SEED"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Train] Device : {device}")
    if device.type == "cuda":
        print(f"[Train] GPU    : {torch.cuda.get_device_name(0)}")
        print(f"[Train] VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        torch.backends.cudnn.benchmark = True

    os.makedirs(CFG["SAVE_DIR"], exist_ok=True)
    ROOT = CFG["ROOT"]

    train_tf = build_train_transform(CFG["IMG_SIZE"])
    val_tf   = build_val_transform(CFG["IMG_SIZE"])

    print("\n[Train] Building datasets …")
    train_ds = ChestXrayDataset(
        csv_file  = f"{ROOT}/Data_Entry_2017.csv",
        root_dir  = ROOT,
        file_list = f"{ROOT}/train_val_list.txt",
        transform = train_tf,
        img_size  = CFG["IMG_SIZE"],
    )
    test_ds = ChestXrayDataset(
        csv_file  = f"{ROOT}/Data_Entry_2017.csv",
        root_dir  = ROOT,
        file_list = f"{ROOT}/test_list.txt",
        transform = val_tf,
        img_size  = CFG["IMG_SIZE"],
    )

    pos_weight = train_ds.compute_pos_weight(clamp_max=CFG["POS_WEIGHT_CAP"]).to(device)
    print("\n[Train] Disease-wise pos_weight:")
    for name, pw in zip(DISEASE_LABELS, pos_weight.cpu()):
        print(f"  {name:<22} {pw:.2f}")

    train_loader = DataLoader(
        train_ds,
        batch_size      = CFG["BATCH_SIZE"],
        shuffle         = True,
        num_workers     = CFG["NUM_WORKERS"],
        pin_memory      = CFG["PIN_MEMORY"],
        persistent_workers = True,
        prefetch_factor = CFG["PREFETCH"],
        drop_last       = True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size      = CFG["BATCH_SIZE"] * 2,
        shuffle         = False,
        num_workers     = CFG["NUM_WORKERS"],
        pin_memory      = CFG["PIN_MEMORY"],
        persistent_workers = True,
        prefetch_factor = CFG["PREFETCH"],
    )

    print("\n[Train] Building model …")
    model = build_model(num_classes=14, dropout=CFG["DROPOUT"])
    model = model.to(device)
    total_p     = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Train] Parameters: {total_p:,} total | {trainable_p:,} trainable")

    criterion = FocalBCELoss(
        pos_weight   = pos_weight,
        gamma        = CFG["FOCAL_GAMMA"],
        label_smooth = CFG["LABEL_SMOOTH"],
    )

    optimizer = build_optimizer(model, CFG)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr          = [CFG["LR_BACKBONE"], CFG["LR_HEAD"]],
        steps_per_epoch = len(train_loader),
        epochs          = CFG["EPOCHS"],
        pct_start       = 0.10,
        anneal_strategy = "cos",
        div_factor      = 10.0,
        final_div_factor= 1e3,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    best_auc = 0.0
    counter  = 0
    metrics  = []
    t0_total = time.time()

    print(f"\n[Train] Starting — {CFG['EPOCHS']} epochs, patience={CFG['PATIENCE']}")
    print(f"[Train] Input: {CFG['IMG_SIZE']}×{CFG['IMG_SIZE']}  |  Batch: {CFG['BATCH_SIZE']}  |  "
          f"TTA: {CFG['TTA_PASSES']}-pass  |  Gamma={CFG['FOCAL_GAMMA']}  |  "
          f"LR_HEAD={CFG['LR_HEAD']}  |  Dropout={CFG['DROPOUT']}\n")

    for epoch in range(1, CFG["EPOCHS"] + 1):
        t0 = time.time()
        model.train()
        running_loss = 0.0
        n_batches    = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{CFG['EPOCHS']}"):
            # images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits = model(images)
                loss   = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item()
            n_batches    += 1

        train_loss = running_loss / n_batches
        epoch_time = time.time() - t0

        val_auc, all_labels, all_probs = evaluate(model, test_loader, device, tta_passes=CFG["TTA_PASSES"])

        per_class_auc = {}
        for i, disease in enumerate(DISEASE_LABELS):
            try:
                per_class_auc[disease] = roc_auc_score(all_labels[:, i], all_probs[:, i])
            except Exception:
                per_class_auc[disease] = float("nan")

        lr_bb = optimizer.param_groups[0]["lr"]
        lr_hd = optimizer.param_groups[1]["lr"]
        print(
            f"Epoch {epoch:02d} | Loss {train_loss:.4f} | AUC {val_auc:.4f} | "
            f"LR backbone={lr_bb:.2e} head={lr_hd:.2e} | Time {epoch_time:.0f}s"
        )
        print(
            f"         Infiltration={per_class_auc.get('Infiltration',0):.4f}  "
            f"Pneumonia={per_class_auc.get('Pneumonia',0):.4f}  "
            f"Consolidation={per_class_auc.get('Consolidation',0):.4f}"
        )

        metrics.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_auc": val_auc,
            "lr_backbone": lr_bb,
            "lr_head": lr_hd,
            **{f"auc_{d}": per_class_auc[d] for d in DISEASE_LABELS},
        })

        if val_auc > best_auc:
            best_auc = val_auc
            counter  = 0
            state = model.state_dict()
            torch.save({
                "epoch":            epoch,
                "auc":              val_auc,
                "model_state_dict": state,
                "per_class_auc":    per_class_auc,
                "cfg":              CFG,
            }, SAVE_PATH)
            print(f"  ✔ Best model saved  AUC={val_auc:.4f}")
        else:
            counter += 1
            print(f"  No improvement ({counter}/{CFG['PATIENCE']})")
            if counter >= CFG["PATIENCE"]:
                print("[Train] Early stopping triggered.")
                break

    total_time = time.time() - t0_total
    print(f"\n{'='*60}")
    print(f"Training complete in {total_time/60:.1f} min")
    print(f"Best macro AUC : {best_auc:.4f}")
    print(f"{'='*60}")

    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(os.path.join(CFG["SAVE_DIR"], "training_metrics.csv"), index=False)

    print("\n[Train] Loading best checkpoint for final evaluation …")
    ckpt = torch.load(SAVE_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    final_auc, all_labels, all_probs = evaluate(model, test_loader, device, tta_passes=CFG["TTA_PASSES"])
    print(f"Final AUC (best ckpt, {CFG['TTA_PASSES']}-pass TTA): {final_auc:.4f}")

    print("\nPer-disease AUC (final):")
    for i, disease in enumerate(DISEASE_LABELS):
        try:
            a = roc_auc_score(all_labels[:, i], all_probs[:, i])
            print(f"  {disease:<22} {a:.4f}")
        except Exception as e:
            print(f"  {disease:<22} ERROR: {e}")

    print("\n[Train] Saving plots …")
    save_training_plots(metrics_df, CFG["SAVE_DIR"])
    save_roc_curves(all_labels, all_probs, CFG["SAVE_DIR"])
    save_per_class_auc(all_labels, all_probs, CFG["SAVE_DIR"])
    save_confusion_matrices(all_labels, all_probs, CFG["SAVE_DIR"], threshold=0.35)

    print("\nClassification Report (threshold=0.35):")
    threshold = 0.35
    for i, disease in enumerate(DISEASE_LABELS):
        y_true = all_labels[:, i]
        y_pred = (all_probs[:, i] >= threshold).astype(int)
        print(f"\n── {disease} ──")
        print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    print("\n[Train] All outputs saved to:", CFG["SAVE_DIR"])
    print(f"[Train] Best AUC = {best_auc:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
# ================================================================
#  Train_T4.py  —  NIH Chest X-Ray14  |  Experiment Run
#  Kaggle T4 (16GB) edition
#
#
#  KEY CHANGES vs the H100 script (and why):
#  ─────────────────────────────────────────────────────────────
#  1. ConvNeXt-Base -> ConvNeXt-SMALL (still .fb_in22k_ft_in1k).
#     You KEEP the ImageNet-21K pretraining — the single biggest
#     quality lever — but at 50M params instead of 88M so it fits.
#  2. 448px -> 384px input. 448 at batch 64 needs ~28GB; impossible
#     on a T4. 384 keeps most of the fine-detail benefit.
#  3. batch 64 -> 16 + GRAD ACCUMULATION x2 (effective batch 32).
#  4. GRADIENT CHECKPOINTING on — trades ~20% speed for big VRAM
#     savings so the model fits with headroom.
#  5. Per-epoch validation = 1-pass, on a fixed 8k SUBSET of test.
#     Full 4-pass TTA on the WHOLE test set runs ONCE at the end.
#     This is what keeps the run inside 12 hours on a slow T4.
#  6. EPOCHS 30 -> 18. Every prior run peaked by epoch 8-10; the
#     data ceiling means more epochs just memorise noise.
#  7. num_workers 8 -> 2 (Kaggle GPU notebooks have ~4 vCPUs).
#  8. TF32 flags removed — Turing (T4) has no TF32; they're no-ops.
#
#  REALISTIC TARGET on T4: 0.83 - 0.86 macro AUC (single model).
#  Deploy the BEST EMA checkpoint.
#
#  MODEL SWITCHER: change CFG["MODEL_NAME"] if you want:
#    - "convnext_tiny.fb_in22k_ft_in1k"   (28M, fastest/safest on time)
#    - "convnext_small.fb_in22k_ft_in1k"  (50M, DEFAULT — best balance)
#    - "convnext_base.fb_in22k_ft_in1k"   (88M, ONLY on T4 x2 / longer)
#
#  Run order:
#    Cell 1:  !pip install timm albumentations -q
#    Cell 2:  paste this entire file, run
# ================================================================

import os
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import glob
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report,
    precision_score, f1_score, recall_score,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2


# =============================================================
#  CONSTANTS
# =============================================================
DISEASE_LABELS = [
    "Atelectasis", "Consolidation", "Infiltration",
    "Pneumothorax", "Edema", "Emphysema", "Fibrosis",
    "Effusion", "Pneumonia", "Pleural_Thickening",
    "Cardiomegaly", "Nodule", "Mass", "Hernia"
]
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


# =============================================================
#  TRANSFORMS  (384px)
# =============================================================
def build_train_transform(img_size: int = 384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(0.0, 0.04),
            scale=(0.92, 1.08),
            rotate=(-8, 8),
            mode=0,
            p=0.5,
        ),
        A.GridDistortion(num_steps=4, distort_limit=0.08, p=0.25),
        A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=0.5),
        A.RandomBrightnessContrast(
            brightness_limit=0.15, contrast_limit=0.15, p=0.5
        ),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.15),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def build_val_transform(img_size: int = 384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


# =============================================================
#  DATASET
# =============================================================
class ChestXrayDataset(Dataset):
    def __init__(
        self,
        csv_file: str,
        root_dir: str,
        file_list: str,
        transform=None,
        img_size: int = 384,
    ):
        self.root_dir  = root_dir
        self.img_size  = img_size
        self.transform = transform

        print("[Dataset] Loading CSV …")
        df = pd.read_csv(csv_file)

        print("[Dataset] Loading file list …")
        with open(file_list) as f:
            self.image_names = [l.strip() for l in f if l.strip()]

        print("[Dataset] Indexing image files …")
        image_paths = glob.glob(
            os.path.join(root_dir, "**", "*.png"), recursive=True
        )
        self.image_dict = {os.path.basename(p): p for p in image_paths}
        print(f"[Dataset] Found {len(self.image_dict):,} PNG files")

        label_map    = dict(zip(df["Image Index"], df["Finding Labels"]))
        patient_map  = dict(zip(df["Image Index"], df["Patient ID"])) \
                       if "Patient ID" in df.columns else None
        label_to_idx = {lbl: i for i, lbl in enumerate(DISEASE_LABELS)}

        print("[Dataset] Encoding labels …")
        n = len(self.image_names)
        label_matrix = np.zeros((n, len(DISEASE_LABELS)), dtype=np.float32)
        self.patient_ids = []

        for row_idx, img_name in enumerate(self.image_names):
            raw = label_map.get(img_name, "No Finding")
            for disease in raw.split("|"):
                col = label_to_idx.get(disease)
                if col is not None:
                    label_matrix[row_idx, col] = 1.0
            if patient_map is not None:
                self.patient_ids.append(
                    str(patient_map.get(img_name, img_name.split("_")[0]))
                )
            else:
                self.patient_ids.append(img_name.split("_")[0])

        self.labels      = torch.from_numpy(label_matrix)
        self.labels_list = DISEASE_LABELS
        n_patients = len(set(self.patient_ids))
        print(f"[Dataset] Ready — {n:,} samples | {n_patients:,} unique patients")

    def compute_pos_weight(self, clamp_max: float = 30.0) -> torch.Tensor:
        pos = self.labels.sum(dim=0)
        neg = len(self.labels) - pos
        pw  = neg / (pos + 1e-6)
        return pw.clamp(min=1.0, max=clamp_max)

    def __len__(self) -> int:
        return len(self.image_names)

    def __getitem__(self, idx: int):
        img_name = self.image_names[idx]
        img_path = self.image_dict.get(img_name)

        if img_path is None:
            img_np = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            try:
                with Image.open(img_path) as pil_img:
                    img_np = np.array(pil_img.convert("RGB"), dtype=np.uint8)
            except Exception:
                img_np = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)

        if self.transform is not None:
            img_tensor = self.transform(image=img_np)["image"]
        else:
            pil = Image.fromarray(img_np).resize(
                (self.img_size, self.img_size), Image.BILINEAR
            )
            img_tensor = torch.from_numpy(
                np.array(pil, dtype=np.float32) / 255.0
            ).permute(2, 0, 1)

        return img_tensor, self.labels[idx]


def report_patient_leakage(
    train_ds: ChestXrayDataset, test_ds: ChestXrayDataset
) -> None:
    train_pts = set(train_ds.patient_ids)
    test_pts  = set(test_ds.patient_ids)
    overlap   = train_pts & test_pts
    pct       = 100 * len(overlap) / max(len(test_pts), 1)
    print(f"\n[LeakCheck] Train patients : {len(train_pts):,}")
    print(f"[LeakCheck] Test  patients : {len(test_pts):,}")
    print(f"[LeakCheck] Overlap        : {len(overlap):,} ({pct:.1f}% of test)")
    print(f"[LeakCheck] NOTE: NIH official split has known patient overlap — "
          f"reported AUC is vs this split.")


# =============================================================
#  CONFIG  (T4-sized)
# =============================================================
CFG = dict(
    ROOT     = "/kaggle/input/datasets/organizations/nih-chest-xrays/data",
    SAVE_DIR = "/kaggle/working",

    MODEL_NAME = "convnext_small.fb_in22k_ft_in1k",  # see switcher in header

    # ── input ─────────────────────────────────────────────────
    IMG_SIZE   = 384,    # 448 OOMs on a T4; 384 keeps most detail

    # ── training ──────────────────────────────────────────────
    EPOCHS      = 18,    # prior runs peaked by ep 8-10; 18 is plenty
    PATIENCE    = 6,
    BATCH_SIZE  = 16,    # fits T4 16GB at 384px with grad checkpointing
    ACCUM_STEPS = 2,     # effective batch = 16 * 2 = 32

    # ── LR: layer-wise decay across ConvNeXt's 4 stages ───────
    LR_HEAD      = 1e-4,
    LR_BACKBONE  = 2e-5,
    LAYER_DECAY  = 0.80,
    WEIGHT_DECAY = 0.05,

    # ── Asymmetric Loss ───────────────────────────────────────
    ASL_GAMMA_POS = 0.0,
    ASL_GAMMA_NEG = 4.0,
    ASL_CLIP      = 0.05,
    LABEL_SMOOTH  = 0.07,

    DROPOUT        = 0.10,
    DROP_PATH_RATE = 0.10,   # Small uses a lower rate than Base

    # ── EMA ───────────────────────────────────────────────────
    EMA_DECAY = 0.9995,

    # ── TTA / validation strategy ─────────────────────────────
    TTA_PASSES_EVAL  = 1,     # fast per-epoch validation (no TTA)
    TTA_PASSES_FINAL = 4,     # full TTA once, on best checkpoint
    VAL_SUBSET       = 8000,  # per-epoch val subset size (None = full)

    GRAD_CHECKPOINT = True,   # big VRAM saver on T4

    # ── dataloader ────────────────────────────────────────────
    NUM_WORKERS = 2,          # Kaggle GPU notebooks ~4 vCPUs
    PIN_MEMORY  = True,
    PREFETCH    = 2,

    SEED = 42,
)

SAVE_PATH     = os.path.join(CFG["SAVE_DIR"], "convnext_best_raw.pth")
SAVE_PATH_EMA = os.path.join(CFG["SAVE_DIR"], "convnext_best_ema.pth")


# =============================================================
#  ASYMMETRIC LOSS
# =============================================================
class AsymmetricLoss(nn.Module):
    def __init__(
        self,
        gamma_pos:    float = 0.0,
        gamma_neg:    float = 4.0,
        clip:         float = 0.05,
        label_smooth: float = 0.07,
        eps:          float = 1e-8,
    ):
        super().__init__()
        self.gamma_pos    = gamma_pos
        self.gamma_neg    = gamma_neg
        self.clip         = clip
        self.label_smooth = label_smooth
        self.eps          = eps

    def forward(
        self, logits: torch.Tensor, targets: torch.Tensor
    ) -> torch.Tensor:
        if self.label_smooth > 0:
            targets = targets * (1 - self.label_smooth) + 0.5 * self.label_smooth

        probs     = torch.sigmoid(logits)
        probs_neg = (1 - probs + self.clip).clamp(max=1)

        loss_pos = targets       * torch.log(probs.clamp(min=self.eps))
        loss_neg = (1 - targets) * torch.log(probs_neg.clamp(min=self.eps))

        pt0 = probs     * targets
        pt1 = probs_neg * (1 - targets)
        pt  = pt0 + pt1
        gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
        w     = torch.pow(1 - pt, gamma)

        return (-(loss_pos + loss_neg) * w).mean()


# =============================================================
#  EMA
# =============================================================
class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.9995):
        self.ema   = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        for ema_p, p in zip(self.ema.parameters(), model.parameters()):
            ema_p.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
        for ema_b, b in zip(self.ema.buffers(), model.buffers()):
            ema_b.copy_(b)


# =============================================================
#  MODEL — ConvNeXt (ImageNet-21K pretrained via timm)
# =============================================================
def build_model(num_classes: int = 14, cfg: dict = CFG) -> nn.Module:
    model = timm.create_model(
        cfg["MODEL_NAME"],
        pretrained     = True,
        num_classes    = num_classes,
        drop_rate      = cfg["DROPOUT"],
        drop_path_rate = cfg["DROP_PATH_RATE"],
    )
    if cfg.get("GRAD_CHECKPOINT", False):
        try:
            model.set_grad_checkpointing(enable=True)
            print("[Model] Gradient checkpointing ENABLED")
        except Exception as e:
            print(f"[Model] Could not enable grad checkpointing: {e}")
    return model


def build_optimizer(model: nn.Module, cfg: dict) -> torch.optim.Optimizer:
    no_decay_kws = ["norm", "bias"]
    stage_names  = ["stem", "stages.0", "stages.1", "stages.2", "stages.3", "head"]
    n_stages     = len(stage_names)
    param_groups = {}

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        stage_idx = n_stages - 1
        for i, sname in enumerate(stage_names):
            if sname in name:
                stage_idx = i
                break

        lr_scale = cfg["LAYER_DECAY"] ** (n_stages - 1 - stage_idx)
        lr       = cfg["LR_BACKBONE"] + (cfg["LR_HEAD"] - cfg["LR_BACKBONE"]) * lr_scale
        no_decay = any(k in name for k in no_decay_kws)
        key      = (round(lr, 10), no_decay)
        if key not in param_groups:
            param_groups[key] = []
        param_groups[key].append(param)

    return torch.optim.AdamW([
        {
            "params":       params,
            "lr":           lr,
            "weight_decay": 0.0 if no_decay else cfg["WEIGHT_DECAY"],
        }
        for (lr, no_decay), params in param_groups.items()
    ])


# =============================================================
#  TTA + EVALUATION
# =============================================================
@torch.no_grad()
def tta_predict(
    model: nn.Module, images: torch.Tensor, n_passes: int = 1
) -> torch.Tensor:
    preds = torch.sigmoid(model(images))
    if n_passes >= 2:
        preds = preds + torch.sigmoid(model(torch.flip(images, dims=[3])))
    if n_passes >= 4:
        H    = images.shape[2]
        crop = int(H * 0.90)
        pad  = (H - crop) // 2
        c1   = images[:, :, pad:pad+crop, pad:pad+crop]
        c1   = F.interpolate(c1, size=(H, H), mode="bilinear", align_corners=False)
        preds = preds + torch.sigmoid(model(c1))
        preds = preds + torch.sigmoid(model(torch.flip(c1, dims=[3])))
    return preds / n_passes


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    tta_passes: int = 1,
    desc: str = "eval",
):
    model.eval()
    all_labels, all_probs = [], []
    for images, labels in tqdm(loader, desc=f"  {desc}", leave=False):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            probs = tta_predict(model, images, n_passes=tta_passes)
        all_labels.append(labels.numpy())
        all_probs.append(probs.float().cpu().numpy())
    all_labels = np.vstack(all_labels)
    all_probs  = np.vstack(all_probs)
    try:
        macro_auc = roc_auc_score(all_labels, all_probs, average="macro")
    except Exception:
        macro_auc = 0.0
    return macro_auc, all_labels, all_probs


# =============================================================
#  PLOTTING
# =============================================================
def save_training_plots(metrics_df: pd.DataFrame, save_dir: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(metrics_df["epoch"], metrics_df["train_loss"],
                 marker="o", label="Train Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss"); axes[0].grid(True)

    axes[1].plot(metrics_df["epoch"], metrics_df["val_auc"],
                 marker="o", color="steelblue", label="Raw AUC")
    axes[1].plot(metrics_df["epoch"], metrics_df["val_auc_ema"],
                 marker="s", color="purple", linewidth=2, label="EMA AUC")
    axes[1].axhline(y=0.86, color="red", linestyle="--", label="Target 0.86")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro AUC")
    axes[1].set_title("Validation AUC — Raw vs EMA")
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_roc_curves(
    labels: np.ndarray, probs: np.ndarray, save_dir: str, suffix: str = ""
) -> None:
    fig, ax = plt.subplots(figsize=(11, 9))
    for i, disease in enumerate(DISEASE_LABELS):
        try:
            fpr, tpr, _ = roc_curve(labels[:, i], probs[:, i])
            roc_auc     = auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f"{disease} ({roc_auc:.3f})")
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(f"ROC Curves — NIH Chest X-ray14{suffix}")
    ax.legend(fontsize=8); ax.grid(True)
    plt.savefig(os.path.join(save_dir, f"roc_curves{suffix}.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[Plot] roc_curves{suffix}.png saved")


def save_per_class_auc(
    labels: np.ndarray, probs: np.ndarray, save_dir: str, suffix: str = ""
) -> None:
    aucs, names = [], []
    for i, disease in enumerate(DISEASE_LABELS):
        try:
            a = roc_auc_score(labels[:, i], probs[:, i])
            aucs.append(a); names.append(disease)
        except Exception:
            pass
    pairs   = sorted(zip(aucs, names))
    aucs_s  = [p[0] for p in pairs]
    names_s = [p[1] for p in pairs]
    colors  = ["#E74C3C" if a < 0.75 else "#F39C12" if a < 0.83 else "#27AE60"
               for a in aucs_s]
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(names_s, aucs_s, color=colors)
    ax.axvline(x=0.86, color="navy", linestyle="--", label="Target 0.86")
    for bar, val in zip(bars, aucs_s):
        ax.text(bar.get_width() + 0.003,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)
    ax.set_xlim(0.60, 1.0); ax.set_xlabel("AUC")
    ax.set_title(f"Per-Disease AUC{suffix}")
    ax.legend(); ax.grid(axis="x", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"per_disease_auc{suffix}.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[Plot] per_disease_auc{suffix}.png saved")


def save_confusion_matrices_grid(
    labels: np.ndarray, probs: np.ndarray, save_dir: str,
    threshold: float = 0.35, suffix: str = ""
) -> None:
    n_cols = 4
    n_rows = int(np.ceil(len(DISEASE_LABELS) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(4 * n_cols, 3.6 * n_rows))
    axes = axes.flatten()
    for i, disease in enumerate(DISEASE_LABELS):
        ax = axes[i]
        try:
            y_true = labels[:, i]
            y_pred = (probs[:, i] >= threshold).astype(int)
            cm     = confusion_matrix(y_true, y_pred)
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                        cbar=False, square=True,
                        xticklabels=["Neg", "Pos"],
                        yticklabels=["Neg", "Pos"],
                        annot_kws={"size": 11})
            d_auc = roc_auc_score(y_true, probs[:, i]) \
                    if len(np.unique(y_true)) > 1 else float("nan")
            ax.set_title(f"{disease}\nAUC={d_auc:.3f}", fontsize=11)
            ax.set_xlabel("Predicted", fontsize=9)
            ax.set_ylabel("Actual",    fontsize=9)
            ax.tick_params(labelsize=9)
        except Exception:
            ax.text(0.5, 0.5, disease, ha="center", va="center")
            ax.axis("off")
    for j in range(len(DISEASE_LABELS), len(axes)):
        axes[j].axis("off")
    fig.suptitle(
        f"Confusion Matrices — All 14 Diseases (threshold={threshold}){suffix}",
        fontsize=14, y=1.01,
    )
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"confusion_matrices_grid{suffix}.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[Plot] confusion_matrices_grid{suffix}.png saved")


def save_precision_recall_f1_curves(
    labels: np.ndarray, probs: np.ndarray, save_dir: str
) -> None:
    thresholds  = np.linspace(0.05, 0.95, 19)
    n_cols      = 4
    n_rows      = int(np.ceil(len(DISEASE_LABELS) / n_cols))
    fig, axes   = plt.subplots(n_rows, n_cols,
                                figsize=(4.2 * n_cols, 3.4 * n_rows))
    axes        = axes.flatten()
    macro_p     = np.zeros(len(thresholds))
    macro_r     = np.zeros(len(thresholds))
    macro_f     = np.zeros(len(thresholds))

    for i, disease in enumerate(DISEASE_LABELS):
        y_true = labels[:, i]
        ps, rs, fs = [], [], []
        for t in thresholds:
            y_pred = (probs[:, i] >= t).astype(int)
            ps.append(precision_score(y_true, y_pred, zero_division=0))
            rs.append(recall_score(y_true,    y_pred, zero_division=0))
            fs.append(f1_score(y_true,        y_pred, zero_division=0))
        macro_p += np.array(ps)
        macro_r += np.array(rs)
        macro_f += np.array(fs)
        ax = axes[i]
        ax.plot(thresholds, ps, color="#2196F3", linewidth=1.6, label="Precision")
        ax.plot(thresholds, rs, color="#FF9800", linewidth=1.6, label="Recall")
        ax.plot(thresholds, fs, color="#4CAF50", linewidth=2.0, label="F1")
        best_idx = int(np.argmax(fs))
        ax.axvline(thresholds[best_idx], color="grey", linestyle=":", linewidth=1)
        ax.scatter([thresholds[best_idx]], [fs[best_idx]],
                   color="#4CAF50", zorder=5, s=25)
        ax.set_title(
            f"{disease}  (F1={fs[best_idx]:.3f} @ t={thresholds[best_idx]:.2f})",
            fontsize=10)
        ax.set_ylim(0, 1); ax.tick_params(labelsize=8); ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7, loc="upper right")

    for j in range(len(DISEASE_LABELS), len(axes)):
        axes[j].axis("off")
    fig.suptitle("Precision / Recall / F1 vs Threshold — Per Disease",
                 fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "prf1_per_class.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

    n = len(DISEASE_LABELS)
    macro_p /= n; macro_r /= n; macro_f /= n
    best_idx = int(np.argmax(macro_f))
    fig2, ax2 = plt.subplots(figsize=(8, 5.5))
    ax2.plot(thresholds, macro_p, label="Macro Precision", color="#2196F3", linewidth=2)
    ax2.plot(thresholds, macro_r, label="Macro Recall",    color="#FF9800", linewidth=2)
    ax2.plot(thresholds, macro_f, label="Macro F1",        color="#4CAF50", linewidth=2.5)
    ax2.axvline(thresholds[best_idx], color="red", linestyle="--", linewidth=1.5,
                label=f"Best F1 @ t={thresholds[best_idx]:.2f}")
    ax2.set_xlabel("Threshold"); ax2.set_ylabel("Score")
    ax2.set_title(
        f"Macro Precision / Recall / F1 vs Threshold\n"
        f"Best Macro F1 = {macro_f[best_idx]:.4f} @ threshold = {thresholds[best_idx]:.2f}"
    )
    ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "prf1_macro.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"[Plot] prf1_per_class.png + prf1_macro.png saved "
          f"(best macro F1={macro_f[best_idx]:.4f} @ t={thresholds[best_idx]:.2f})")


# =============================================================
#  MAIN
# =============================================================
def main():
    torch.manual_seed(CFG["SEED"])
    np.random.seed(CFG["SEED"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Train] Device : {device}")
    if device.type == "cuda":
        print(f"[Train] GPU    : {torch.cuda.get_device_name(0)}")
        print(f"[Train] VRAM   : "
              f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        torch.backends.cudnn.benchmark = True
        # NOTE: TF32 is Ampere+; the T4 (Turing) has none, so no TF32 flags here.

    os.makedirs(CFG["SAVE_DIR"], exist_ok=True)
    ROOT = CFG["ROOT"]

    train_tf = build_train_transform(CFG["IMG_SIZE"])
    val_tf   = build_val_transform(CFG["IMG_SIZE"])

    print("\n[Train] Building datasets …")
    train_ds = ChestXrayDataset(
        csv_file  = f"{ROOT}/Data_Entry_2017.csv",
        root_dir  = ROOT,
        file_list = f"{ROOT}/train_val_list.txt",
        transform = train_tf,
        img_size  = CFG["IMG_SIZE"],
    )
    test_ds = ChestXrayDataset(
        csv_file  = f"{ROOT}/Data_Entry_2017.csv",
        root_dir  = ROOT,
        file_list = f"{ROOT}/test_list.txt",
        transform = val_tf,
        img_size  = CFG["IMG_SIZE"],
    )
    report_patient_leakage(train_ds, test_ds)

    # ── fast per-epoch validation subset (fixed across epochs) ──
    if CFG["VAL_SUBSET"] is not None and CFG["VAL_SUBSET"] < len(test_ds):
        rng        = np.random.RandomState(CFG["SEED"])
        sub_idx    = rng.choice(len(test_ds), CFG["VAL_SUBSET"], replace=False)
        val_ds_fast = Subset(test_ds, sub_idx.tolist())
        print(f"[Train] Per-epoch val subset: {len(val_ds_fast):,} "
              f"of {len(test_ds):,} test images")
    else:
        val_ds_fast = test_ds

    train_loader = DataLoader(
        train_ds,
        batch_size         = CFG["BATCH_SIZE"],
        shuffle            = True,
        num_workers        = CFG["NUM_WORKERS"],
        pin_memory         = CFG["PIN_MEMORY"],
        persistent_workers = True,
        prefetch_factor    = CFG["PREFETCH"],
        drop_last          = True,
    )
    val_loader_fast = DataLoader(
        val_ds_fast,
        batch_size         = CFG["BATCH_SIZE"] * 2,
        shuffle            = False,
        num_workers        = CFG["NUM_WORKERS"],
        pin_memory         = CFG["PIN_MEMORY"],
        persistent_workers = True,
        prefetch_factor    = CFG["PREFETCH"],
    )
    test_loader_full = DataLoader(
        test_ds,
        batch_size         = CFG["BATCH_SIZE"] * 2,
        shuffle            = False,
        num_workers        = CFG["NUM_WORKERS"],
        pin_memory         = CFG["PIN_MEMORY"],
        persistent_workers = False,
        prefetch_factor    = CFG["PREFETCH"],
    )

    print(f"\n[Train] Building {CFG['MODEL_NAME']} …")
    model = build_model(num_classes=14, cfg=CFG)
    model = model.to(device)
    total_p     = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Train] Parameters: {total_p:,} total | {trainable_p:,} trainable")

    ema       = ModelEMA(model, decay=CFG["EMA_DECAY"])
    criterion = AsymmetricLoss(
        gamma_pos    = CFG["ASL_GAMMA_POS"],
        gamma_neg    = CFG["ASL_GAMMA_NEG"],
        clip         = CFG["ASL_CLIP"],
        label_smooth = CFG["LABEL_SMOOTH"],
    )
    optimizer = build_optimizer(model, CFG)
    print(f"[Train] Optimiser: {len(optimizer.param_groups)} param groups "
          f"(layer-wise LR decay={CFG['LAYER_DECAY']})")

    # T_0 chosen so a warm restart lands roughly mid-run
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0     = max(1, CFG["EPOCHS"] // 2),
        T_mult  = 1,
        eta_min = 1e-7,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    best_auc     = 0.0
    best_auc_ema = 0.0
    counter      = 0
    metrics      = []
    accum_steps  = max(1, CFG["ACCUM_STEPS"])
    t0_total     = time.time()

    print(f"\n[Train] ═══════════════════════════════════════")
    print(f"[Train] {CFG['MODEL_NAME']}")
    print(f"[Train] Input {CFG['IMG_SIZE']}×{CFG['IMG_SIZE']} | "
          f"Batch {CFG['BATCH_SIZE']}×{accum_steps}accum "
          f"(eff {CFG['BATCH_SIZE']*accum_steps})")
    print(f"[Train] Epochs {CFG['EPOCHS']} | Patience {CFG['PATIENCE']}")
    print(f"[Train] Val: {CFG['TTA_PASSES_EVAL']}-pass on subset | "
          f"Final: {CFG['TTA_PASSES_FINAL']}-pass on full test")
    print(f"[Train] ASL γ_neg={CFG['ASL_GAMMA_NEG']} | "
          f"clip={CFG['ASL_CLIP']} | smooth={CFG['LABEL_SMOOTH']}")
    print(f"[Train] EMA decay={CFG['EMA_DECAY']}")
    print(f"[Train] ═══════════════════════════════════════\n")

    for epoch in range(1, CFG["EPOCHS"] + 1):
        t0 = time.time()
        model.train()
        running_loss = 0.0
        n_batches    = 0
        optimizer.zero_grad(set_to_none=True)

        n_steps = len(train_loader)
        for step, (images, labels) in enumerate(tqdm(
            train_loader, desc=f"Epoch {epoch:02d}/{CFG['EPOCHS']}"
        )):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits = model(images)
                loss   = criterion(logits, labels) / accum_steps

            scaler.scale(loss).backward()

            is_step = ((step + 1) % accum_steps == 0) or ((step + 1) == n_steps)
            if is_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)

            running_loss += loss.item() * accum_steps
            n_batches    += 1

        scheduler.step()

        train_loss = running_loss / n_batches
        epoch_time = time.time() - t0

        val_auc, _, _ = evaluate(
            model, val_loader_fast, device,
            CFG["TTA_PASSES_EVAL"], "raw")
        val_auc_ema, all_labels, all_probs_ema = evaluate(
            ema.ema, val_loader_fast, device,
            CFG["TTA_PASSES_EVAL"], "ema")

        per_class_auc = {}
        for i, d in enumerate(DISEASE_LABELS):
            try:
                per_class_auc[d] = roc_auc_score(
                    all_labels[:, i], all_probs_ema[:, i]
                )
            except Exception:
                per_class_auc[d] = float("nan")

        print(
            f"Epoch {epoch:02d} | Loss {train_loss:.5f} | "
            f"AUC(Raw)={val_auc:.4f}  AUC(EMA)={val_auc_ema:.4f} | "
            f"Time {epoch_time:.0f}s"
        )
        print(
            f"         Infiltration={per_class_auc.get('Infiltration',0):.4f}  "
            f"Pneumonia={per_class_auc.get('Pneumonia',0):.4f}  "
            f"Consolidation={per_class_auc.get('Consolidation',0):.4f}"
        )

        metrics.append({
            "epoch": epoch, "train_loss": train_loss,
            "val_auc": val_auc, "val_auc_ema": val_auc_ema,
            **{f"auc_{d}": per_class_auc[d] for d in DISEASE_LABELS},
        })

        improved = False

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save({
                "epoch": epoch, "auc": val_auc,
                "model_state_dict": model.state_dict(),
                "per_class_auc": per_class_auc, "cfg": CFG,
            }, SAVE_PATH)
            improved = True
            print(f"  ✔ Best RAW  saved  AUC={val_auc:.4f}")

        if val_auc_ema > best_auc_ema:
            best_auc_ema = val_auc_ema
            torch.save({
                "epoch": epoch, "auc": val_auc_ema,
                "model_state_dict": ema.ema.state_dict(),
                "per_class_auc": per_class_auc, "cfg": CFG,
            }, SAVE_PATH_EMA)
            improved = True
            print(f"  ✔ Best EMA  saved  AUC={val_auc_ema:.4f}")

        if improved:
            counter = 0
        else:
            counter += 1
            print(f"  No improvement ({counter}/{CFG['PATIENCE']})")
            if counter >= CFG["PATIENCE"]:
                print("[Train] Early stopping.")
                break

    # ── Final summary ─────────────────────────────────────────
    total_time = time.time() - t0_total
    print(f"\n{'='*60}")
    print(f"Training complete in {total_time/60:.1f} min")
    print(f"Best RAW AUC (subset) : {best_auc:.4f}")
    print(f"Best EMA AUC (subset) : {best_auc_ema:.4f}")
    print(f"{'='*60}")

    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(
        os.path.join(CFG["SAVE_DIR"], "training_metrics.csv"), index=False
    )

    # ── Final evaluation: best checkpoint, FULL test set, full TTA ──
    best_path = SAVE_PATH_EMA if best_auc_ema >= best_auc else SAVE_PATH
    print(f"\n[Train] Loading {best_path} for final FULL-test evaluation …")
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    final_auc_val, all_labels, all_probs = evaluate(
        model, test_loader_full, device, CFG["TTA_PASSES_FINAL"], "final"
    )
    print(f"\nFINAL macro AUC (full test, {CFG['TTA_PASSES_FINAL']}-pass TTA): "
          f"{final_auc_val:.4f}")

    print("\nPer-disease AUC (final, full test):")
    for i, disease in enumerate(DISEASE_LABELS):
        try:
            a = roc_auc_score(all_labels[:, i], all_probs[:, i])
            print(f"  {disease:<22} {a:.4f}")
        except Exception as e:
            print(f"  {disease:<22} ERROR: {e}")

    # ── Save all plots ─────────────────────────────────────────
    print("\n[Train] Saving plots …")
    save_training_plots(metrics_df, CFG["SAVE_DIR"])
    save_roc_curves(all_labels, all_probs, CFG["SAVE_DIR"])
    save_per_class_auc(all_labels, all_probs, CFG["SAVE_DIR"])
    save_confusion_matrices_grid(
        all_labels, all_probs, CFG["SAVE_DIR"], threshold=0.35
    )
    save_precision_recall_f1_curves(all_labels, all_probs, CFG["SAVE_DIR"])

    # ── Classification report ──────────────────────────────────
    threshold = 0.35
    print(f"\nClassification Report (threshold={threshold}):")
    for i, disease in enumerate(DISEASE_LABELS):
        y_true = all_labels[:, i]
        y_pred = (all_probs[:, i] >= threshold).astype(int)
        print(f"\n── {disease} ──")
        print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    print("\n[Train] All outputs saved to:", CFG["SAVE_DIR"])
    print(f"[Train] FINAL full-test AUC = {final_auc_val:.4f}")
    print(f"[Train] Deploy checkpoint:    {best_path}")


if __name__ == "__main__":
    main()

In [ ]:
# =============================================================
#  Evaluate.py  —  NIH Chest X-Ray14  |  Accuracy report
#  Matches "Tuned Run #2" (Config 2): 384px, 4-pass TTA.
#
#  What it reports:
#    1. Image-level EXACT-MATCH accuracy  (all 14 findings correct)
#         -> "X-rays detected fully correct" vs "wrong on >=1 finding"
#    2. Per-label (Hamming) accuracy       (correct decisions / N*14)
#    3. Per-class accuracy + TP/FP/FN/TN + precision/recall/F1 + AUC
#    4. Macro AUC / F1 / precision / recall
#
#  Run order on Kaggle:
#    Cell 1:  !pip install albumentations -q
#    Cell 2:  paste this file, set EVAL_CFG["CKPT_PATH"], run
#
#  IMPORTANT (honesty for the paper):
#    - AUC is the primary metric; accuracy is imbalance-sensitive.
#    - If USE_PER_CLASS_THRESHOLDS=True, thresholds are tuned on the
#      TEST set here for convenience; for a paper, tune them on a
#      validation split instead and state that clearly.
# =============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from PIL import Image
import glob
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
)

import albumentations as A
from albumentations.pytorch import ToTensorV2


# =============================================================
#  CONSTANTS
# =============================================================
DISEASE_LABELS = [
    "Atelectasis", "Consolidation", "Infiltration",
    "Pneumothorax", "Edema", "Emphysema", "Fibrosis",
    "Effusion", "Pneumonia", "Pleural_Thickening",
    "Cardiomegaly", "Nodule", "Mass", "Hernia"
]
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


# =============================================================
#  CONFIG
# =============================================================
EVAL_CFG = dict(
    ROOT      = "/kaggle/input/datasets/organizations/nih-chest-xrays/data",
    # Point this at the checkpoint your training saved.
    # Config-2 DenseNet201 run saved "DenseNet201.pth".
    CKPT_PATH = "/kaggle/input/models/argha2004/efficientnetv2-snew-config/pytorch/default/1/EfficientNetV2-S.pth",
    SAVE_DIR  = "/kaggle/working",

    MODEL     = "efficientnet_v2_s",   # "densenet201" | "densenet121" | "efficientnet_v2_s"
    IMG_SIZE  = 384,
    BATCH_SIZE = 64,
    TTA_PASSES = 4,              # set 1 for a fast (no-TTA) pass
    NUM_WORKERS = 4,

    THRESHOLD = 0.35,                 # global threshold (used if not per-class)
    USE_PER_CLASS_THRESHOLDS = False, # True -> pick each class's best-F1 threshold
    DROPOUT = 0.2,
    SEED = 42,
)


# =============================================================
#  TRANSFORM (val) + DATASET  (same as training)
# =============================================================
def build_val_transform(img_size: int = 384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


class ChestXrayDataset(Dataset):
    def __init__(self, csv_file, root_dir, file_list, transform=None, img_size=384):
        self.root_dir  = root_dir
        self.img_size  = img_size
        self.transform = transform

        print("[Data] Loading CSV …")
        df = pd.read_csv(csv_file)
        print("[Data] Loading file list …")
        with open(file_list) as f:
            self.image_names = [l.strip() for l in f if l.strip()]

        print("[Data] Indexing image files …")
        image_paths = glob.glob(os.path.join(root_dir, "**", "*.png"), recursive=True)
        self.image_dict = {os.path.basename(p): p for p in image_paths}
        print(f"[Data] Found {len(self.image_dict):,} PNG files")

        label_map    = dict(zip(df["Image Index"], df["Finding Labels"]))
        label_to_idx = {lbl: i for i, lbl in enumerate(DISEASE_LABELS)}

        n = len(self.image_names)
        label_matrix = np.zeros((n, len(DISEASE_LABELS)), dtype=np.float32)
        for row_idx, img_name in enumerate(self.image_names):
            raw = label_map.get(img_name, "No Finding")
            for disease in raw.split("|"):
                col = label_to_idx.get(disease)
                if col is not None:
                    label_matrix[row_idx, col] = 1.0
        self.labels = torch.from_numpy(label_matrix)
        print(f"[Data] Ready — {n:,} samples")

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = self.image_dict.get(img_name)
        if img_path is None:
            img_np = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            try:
                with Image.open(img_path) as pil_img:
                    img_np = np.array(pil_img.convert("RGB"), dtype=np.uint8)
            except Exception:
                img_np = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        img_tensor = self.transform(image=img_np)["image"]
        return img_tensor, self.labels[idx]


# =============================================================
#  MODEL  (build the same architecture, then load weights)
# =============================================================
def build_model(name: str, num_classes: int = 14, dropout: float = 0.2) -> nn.Module:
    if name == "densenet201":
        m = models.densenet201(weights=None)
        in_f = m.classifier.in_features
        m.classifier = nn.Sequential(nn.Dropout(p=dropout, inplace=True),
                                     nn.Linear(in_f, num_classes))
    elif name == "densenet121":
        m = models.densenet121(weights=None)
        in_f = m.classifier.in_features
        m.classifier = nn.Sequential(nn.Dropout(p=dropout, inplace=True),
                                     nn.Linear(in_f, num_classes))
    elif name == "efficientnet_v2_s":
        m = models.efficientnet_v2_s(weights=None)
        in_f = m.classifier[1].in_features
        m.classifier = nn.Sequential(nn.Dropout(p=dropout, inplace=True),
                                     nn.Linear(in_f, num_classes))
    else:
        raise ValueError(f"Unknown MODEL: {name}")
    return m


# =============================================================
#  TTA + INFERENCE
# =============================================================
@torch.no_grad()
def tta_predict(model, images, n_passes=4):
    preds = torch.sigmoid(model(images))
    if n_passes >= 2:
        preds = preds + torch.sigmoid(model(torch.flip(images, dims=[3])))
    if n_passes >= 4:
        H = images.shape[2]
        crop = int(H * 0.90)
        pad  = (H - crop) // 2
        c1 = images[:, :, pad:pad+crop, pad:pad+crop]
        c1 = F.interpolate(c1, size=(H, H), mode="bilinear", align_corners=False)
        preds = preds + torch.sigmoid(model(c1))
        preds = preds + torch.sigmoid(model(torch.flip(c1, dims=[3])))
    return preds / n_passes


@torch.no_grad()
def run_inference(model, loader, device, tta_passes=4):
    model.eval()
    all_labels, all_probs = [], []
    for images, labels in tqdm(loader, desc="Inference"):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            probs = tta_predict(model, images, n_passes=tta_passes)
        all_labels.append(labels.numpy())
        all_probs.append(probs.float().cpu().numpy())
    return np.vstack(all_labels), np.vstack(all_probs)


# =============================================================
#  THRESHOLDS + METRICS
# =============================================================
def per_class_best_thresholds(labels, probs, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    thr = np.full(labels.shape[1], 0.35, dtype=np.float32)
    for i in range(labels.shape[1]):
        best_f1, best_t = -1.0, 0.35
        for t in grid:
            yp = (probs[:, i] >= t).astype(int)
            f = f1_score(labels[:, i], yp, zero_division=0)
            if f > best_f1:
                best_f1, best_t = f, t
        thr[i] = best_t
    return thr


def compute_report(labels, probs, thresholds):
    """thresholds: array (14,). Returns dict of everything."""
    N, C = labels.shape
    preds = (probs >= thresholds[None, :]).astype(int)
    labs  = labels.astype(int)

    # ---- image-level exact match ----
    exact_correct = int((preds == labs).all(axis=1).sum())
    exact_wrong   = N - exact_correct

    # ---- per-label (Hamming) ----
    total_decisions = N * C
    correct_decisions = int((preds == labs).sum())

    # ---- per class ----
    rows = []
    for i, d in enumerate(DISEASE_LABELS):
        yt, yp = labs[:, i], preds[:, i]
        tp = int(((yp == 1) & (yt == 1)).sum())
        fp = int(((yp == 1) & (yt == 0)).sum())
        fn = int(((yp == 0) & (yt == 1)).sum())
        tn = int(((yp == 0) & (yt == 0)).sum())
        acc = (tp + tn) / max(N, 1)
        prec = precision_score(yt, yp, zero_division=0)
        rec  = recall_score(yt, yp, zero_division=0)
        f1   = f1_score(yt, yp, zero_division=0)
        try:
            auc_i = roc_auc_score(yt, probs[:, i]) if len(np.unique(yt)) > 1 else float("nan")
        except Exception:
            auc_i = float("nan")
        rows.append(dict(disease=d, thr=float(thresholds[i]), n_pos=int(yt.sum()),
                         TP=tp, FP=fp, FN=fn, TN=tn, accuracy=acc,
                         precision=prec, recall=rec, f1=f1, auc=auc_i))

    macro_auc = np.nanmean([r["auc"] for r in rows])
    macro_f1  = np.mean([r["f1"] for r in rows])
    macro_p   = np.mean([r["precision"] for r in rows])
    macro_r   = np.mean([r["recall"] for r in rows])

    return dict(
        N=N, C=C,
        exact_correct=exact_correct, exact_wrong=exact_wrong,
        exact_acc=exact_correct / max(N, 1),
        correct_decisions=correct_decisions, total_decisions=total_decisions,
        hamming_acc=correct_decisions / max(total_decisions, 1),
        rows=rows,
        macro_auc=macro_auc, macro_f1=macro_f1,
        macro_precision=macro_p, macro_recall=macro_r,
    )


def print_report(rep, threshold_desc):
    print("\n" + "=" * 68)
    print("  NIH ChestX-ray14  —  EVALUATION REPORT")
    print("=" * 68)
    print(f"  Test images        : {rep['N']:,}")
    print(f"  Findings per image : {rep['C']}")
    print(f"  Threshold          : {threshold_desc}")

    print("\n" + "-" * 68)
    print("  1) IMAGE-LEVEL EXACT-MATCH ACCURACY (all 14 findings correct)")
    print("-" * 68)
    print(f"  Fully correct X-rays : {rep['exact_correct']:,} / {rep['N']:,} "
          f"({100*rep['exact_acc']:.2f}%)")
    print(f"  Wrong on >=1 finding : {rep['exact_wrong']:,} / {rep['N']:,} "
          f"({100*(1-rep['exact_acc']):.2f}%)")
    print("  (Strictest metric — hard to score high on 14 simultaneous labels.)")

    print("\n" + "-" * 68)
    print("  2) PER-LABEL (HAMMING) ACCURACY (every disease decision)")
    print("-" * 68)
    print(f"  Correct decisions    : {rep['correct_decisions']:,} / "
          f"{rep['total_decisions']:,} ({100*rep['hamming_acc']:.2f}%)")
    print("  (High partly because most labels are negative — imbalance inflates it.)")

    print("\n" + "-" * 68)
    print("  3) PER-CLASS BREAKDOWN")
    print("-" * 68)
    hdr = (f"  {'Disease':<20}{'thr':>5}{'AUC':>8}{'Acc':>8}"
           f"{'Prec':>8}{'Rec':>8}{'F1':>8}{'TP':>7}{'FP':>7}{'FN':>7}")
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))
    for r in rep["rows"]:
        print(f"  {r['disease']:<20}{r['thr']:>5.2f}{r['auc']:>8.4f}"
              f"{r['accuracy']:>8.4f}{r['precision']:>8.4f}{r['recall']:>8.4f}"
              f"{r['f1']:>8.4f}{r['TP']:>7}{r['FP']:>7}{r['FN']:>7}")

    print("\n" + "-" * 68)
    print("  4) MACRO AVERAGES")
    print("-" * 68)
    print(f"  Macro AUC       : {rep['macro_auc']:.4f}   <- primary metric")
    print(f"  Macro F1        : {rep['macro_f1']:.4f}")
    print(f"  Macro Precision : {rep['macro_precision']:.4f}")
    print(f"  Macro Recall    : {rep['macro_recall']:.4f}")
    print("=" * 68 + "\n")


# =============================================================
#  MAIN
# =============================================================
def main():
    torch.manual_seed(EVAL_CFG["SEED"]); np.random.seed(EVAL_CFG["SEED"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Eval] Device: {device}")
    if device.type == "cuda":
        print(f"[Eval] GPU   : {torch.cuda.get_device_name(0)}")
        torch.backends.cudnn.benchmark = True

    ROOT = EVAL_CFG["ROOT"]
    val_tf = build_val_transform(EVAL_CFG["IMG_SIZE"])

    test_ds = ChestXrayDataset(
        csv_file  = f"{ROOT}/Data_Entry_2017.csv",
        root_dir  = ROOT,
        file_list = f"{ROOT}/test_list.txt",
        transform = val_tf,
        img_size  = EVAL_CFG["IMG_SIZE"],
    )
    test_loader = DataLoader(
        test_ds, batch_size=EVAL_CFG["BATCH_SIZE"], shuffle=False,
        num_workers=EVAL_CFG["NUM_WORKERS"], pin_memory=True,
    )

    print(f"\n[Eval] Building {EVAL_CFG['MODEL']} and loading checkpoint …")
    model = build_model(EVAL_CFG["MODEL"], 14, EVAL_CFG["DROPOUT"]).to(device)
    ckpt = torch.load(EVAL_CFG["CKPT_PATH"], map_location=device, weights_only=False)
    state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:    print(f"[Eval] Missing keys   : {missing}")
    if unexpected: print(f"[Eval] Unexpected keys: {unexpected}")
    if "auc" in ckpt:   print(f"[Eval] Checkpoint stored AUC   : {ckpt['auc']:.4f}")
    if "epoch" in ckpt: print(f"[Eval] Checkpoint stored epoch : {ckpt['epoch']}")

    print(f"\n[Eval] Running inference ({EVAL_CFG['TTA_PASSES']}-pass TTA) …")
    labels, probs = run_inference(model, test_loader, device, EVAL_CFG["TTA_PASSES"])

    if EVAL_CFG["USE_PER_CLASS_THRESHOLDS"]:
        thresholds = per_class_best_thresholds(labels, probs)
        thr_desc = "per-class best-F1 (tuned on test — see header caveat)"
    else:
        thresholds = np.full(14, EVAL_CFG["THRESHOLD"], dtype=np.float32)
        thr_desc = f"global {EVAL_CFG['THRESHOLD']}"

    rep = compute_report(labels, probs, thresholds)
    print_report(rep, thr_desc)

    # ---- save CSV summary ----
    os.makedirs(EVAL_CFG["SAVE_DIR"], exist_ok=True)
    df = pd.DataFrame(rep["rows"])
    summary_path = os.path.join(EVAL_CFG["SAVE_DIR"], "evaluation_per_class_efficientnet.csv")
    df.to_csv(summary_path, index=False)

    overall = pd.DataFrame([{
        "test_images": rep["N"],
        "exact_match_correct": rep["exact_correct"],
        "exact_match_wrong": rep["exact_wrong"],
        "exact_match_accuracy": rep["exact_acc"],
        "hamming_accuracy": rep["hamming_acc"],
        "macro_auc": rep["macro_auc"],
        "macro_f1": rep["macro_f1"],
        "macro_precision": rep["macro_precision"],
        "macro_recall": rep["macro_recall"],
    }])
    overall_path = os.path.join(EVAL_CFG["SAVE_DIR"], "evaluation_overall_efficientnet.csv")
    overall.to_csv(overall_path, index=False)
    print(f"[Eval] Saved: {summary_path}")
    print(f"[Eval] Saved: {overall_path}")


if __name__ == "__main__":
    main()3